In [0]:
%run /Workspace/Users/marcoaurelioreislima@gmail.com/databricks-playground/projects/data_generator/0_data_generator

In [0]:
from rand_engine.main.cdc_generator import CDCGenerator

In [0]:
def reset_env():
    TABLE_ORDER_BRONZE = "prd.l_bronze.orders_retail"
    TABLE_ORDER_SILVER = "prd.l_silver.orders_retail"
    PATH_CHECKPOINTS = "/Volumes/prd/streaming_management/checkpoints"
    CHECKPOINT_BRONZE_ORDER = f"{PATH_CHECKPOINTS}/l_bronze/orders_retail"
    CHECKPOINT_SILVER_ORDER = f"{PATH_CHECKPOINTS}/l_silver/orders_retail"
    PATH_INPUT_CDC = "/Volumes/prd/demo_volumes/rand_engine_data/logs/orders/parquet"
    dbutils.fs.rm(PATH_CHECKPOINTS, True)
    dbutils.fs.rm(PATH_INPUT_CDC, True)
    spark.sql(f"DROP TABLE IF EXISTS {TABLE_ORDER_BRONZE}")
    spark.sql(f"DROP TABLE IF EXISTS {TABLE_ORDER_SILVER}")

# reset_env()

In [0]:
class FakeDigitalOrders:

    def __init__(self):
        self.faker = faker.Faker(locale="pt_BR")

    def metadata(self):
        return {
            "order_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=16)
            },
            "user_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=10)
            },
            "product_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=3)
            },
            "user_type": {     
            "method": DistinctCore.gen_distincts_untyped,
            "parms": dict(distinct=DistinctUtils.handle_distincts_lvl_1({"standard": 80,"premium":15, "gold": 5, None: 7}, 1))
            },
            "device": {
                "method": DistinctCore.gen_distincts_typed,
                "parms": dict(distinct=["IOS", "Android", "Desktop"])
            },
            "traffic_source": {
                "method": DistinctCore.gen_distincts_typed,
                "parms": dict(distinct=["website", "linkedin", "email"])
            }
        }

    def transformer(self):
        def wrapped_transformer(df: PandasDF) -> PandasDF:
            for col in df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
                df[col] = df[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            start_time = dt.now() - timedelta(minutes=10)
            end_time = dt.now()
            random_times = pd.to_datetime(np.random.uniform(start_time.timestamp(), end_time.timestamp(), size=len(df)), unit='s').strftime('%Y-%m-%dT%H:%M:%S')
            df["created_at"] = random_times
            return df
        return wrapped_transformer

In [0]:
def generate_orders_file():
    BASE_PATH = "/Volumes/prd/demo_volumes/rand_engine_data/logs"
    FILE_NAME = "orders"
    FILE_EXT =  "parquet"

    file_generator = FilesGenerator(FakeDigitalOrders()).setup_output(BASE_PATH, file_name=FILE_NAME, ext=FILE_EXT)
    #file_generator.delete_files()
    file_generator.stream_files(size=1000, rounds=2, period=1)
    display(file_generator.list_files())
    df = spark.read.format(FILE_EXT).load(f"{BASE_PATH}/{FILE_NAME}/{FILE_EXT}")
    display(df)
generate_orders_file()

# FilesGenerator(FakeOrders()).generate_sample(100)

### 1. Prototype Batch Mode

Explore the dataset and test out transformation logic using batch dataframes

In [0]:
TABLE_ORDER_BRONZE = "prd.l_bronze.orders_retail"
TABLE_ORDER_SILVER = "prd.l_silver.orders_retail"
PATH_CHECKPOINTS = "/Volumes/prd/streaming_management/checkpoints"
CHECKPOINT_BRONZE_ORDER = f"{PATH_CHECKPOINTS}/l_bronze/orders_retail"
CHECKPOINT_SILVER_ORDER = f"{PATH_CHECKPOINTS}/l_silver/orders_retail"
PATH_INPUT_CDC = "/Volumes/prd/demo_volumes/rand_engine_data/logs/orders/parquet"

In [0]:

from pyspark.sql.functions import col

streaming_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", CHECKPOINT_BRONZE_ORDER)
    .load(PATH_INPUT_CDC)
    .filter(col("traffic_source") == "email")
    .withColumn("mobile", col("device").isin("IOS", "Android"))
    .withColumnRenamed("created_at", "event_timestamp")
    .select("user_id", "event_timestamp", "device", "mobile")
    .writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT_BRONZE_ORDER)
    .toTable(TABLE_ORDER_BRONZE)
    .awaitTermination()
)
# Wait a moment to let the stream initialize
# You can monitor the stream with query.status or query.lastProgress

In [0]:
spark.table(TABLE_ORDER_BRONZE).limit(10).display()

In [0]:
from pyspark.sql.functions import window, sum, col, date_format
from pyspark.sql.types import TimestampType


parsed_df = (
    spark.readStream
    .table(TABLE_ORDER_BRONZE)
    .withColumn("event_timestamp", col("event_timestamp").cast(TimestampType()))
    .withWatermark(eventTime="event_timestamp", delayThreshold="10 minutes")
    .groupBy(window(timeColumn="event_timestamp", windowDuration="20 minutes"), "device")
    .count()
    .withColumn("start_time", date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("end_time", date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss"))
    .drop("window")
    .writeStream
    .format("delta")
    .outputMode("complete")
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT_SILVER_ORDER)
    .toTable(TABLE_ORDER_SILVER)
    .awaitTermination()
)

In [0]:
spark.table(TABLE_ORDER_SILVER).limit(10).display()


## OutputMode = `update`

- Only possible using foreachBatch and MERGE.

In [0]:
stop_streamings():
    for s in spark.streams.active:
        print(s.name)
        s.stop()

In [0]:
# %sql
# -- Row level security
# CREATE FUNCTION hide_rows (region STRING)
# RETURN IF(IS_MEMBER('admin'), true, region="US");

# ALTER TABLE sales SET ROW FILTER hide_rows ON region;

# CREATE OR REPLACE FUNCTION ssn_mask(ssn STRING)
# RETURN CASE WHEN IS_MEMBER('admin') THEN ssn ELSE '*****' END;
# ALTER TABLE users ALTER COLUMN table_ssn SET MASK ssn_mask;


In [0]:
# %sql
# -- What tables are in the catalog
# SELECT table_name
# FROM system.information_schema.tables
# WHERE table_catalog = 'prd';

# -- Who laste updated the gold tables and when
# SELECT table_name, last_altered_by, last_altered
# FROM system.information_schema.tables
# WHERE table_schema = 'l_bronze'
# ORDER BY 1, 3 DESC;

# -- Who has access to this table
# SELECT table_name
# FROM system.information_schema.table_privileges
# WHERE table_name = 'sm_customers';

# -- Who owns this gold table
# SELECT table_owner
# FROM system.information_schema.tables
# WHERE table_catalog = 'retail_prod'
# AND table_schema = 'l_silver'
# AND table_name = 'sm_customers';


# -- Reference http://docs.databricks.com/en/system-tables/index.html

In [0]:
# %sql
# -- What is the daily trend in DBU comsumption
# SELECT usage_date AS `Date`, SUM(usage_quantity) AS `DBU Consumed`
# FROM system.billing.usage
# GROUP BY usage_date
# ORDER BY usage_date ASC;

# -- How many DBUs of each SKU have been used so far this month
# SELECT sku_name AS `SKU`, SUM(usage_quantity) AS `DBUs`
# FROM system.billing.usage
# WHERE month(usage_date) = month(CURRENT_DATE)
# GROUP BY sku
# ORDER BY `DBUs` DESC;

# -- Which 10 users consumed the most DBUs
# SELECT identity_metadata.run_as AS `User`, SUM(usage_quantity) AS `DBUs`
# FROM system.billing.usage
# GROUP BY identity_metadata.run_as 
# ORDER BY `DBUs` DESC
# LIMIT 10;

# -- Which Jobs consumed the most DBUs
# SELECT usage_metadata.job_id AS `Job ID`, SUM(usage_quantity) AS `DBUs`
# FROM system.billing.usage
# GROUP BY identity_metadata.run_as 
# ORDER BY `Job ID`;

# -- Reference http://docs.databricks.com/en/system-tables/billing.html